# Глубинное обучение на табличных данных

Начнем конечно с импортов различных, которые понадобятся далее. Также, многое отсюда было реализовано с помощью функций копипаста с семинаров, что мы проходили, помимо этого, семинары дали вдохновление на именно такой пайплайн действий) Стоило упомянуть

In [1]:
import pandas as pd
import numpy as np
import random

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader #чтобы подавать данные батчами

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score #для подсчета качества

Зафиксируем сиды

In [2]:
def seed_everything(seed=42): # чтобы запуски были стабильными и воспроизводимымми
    random.seed(seed) # фиксируем генератор случайных чисел
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    
seed_everything(42)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device # то есть, если есть видеокарта то юзаем cuda, если нет - то юзаем cpu

device(type='cpu')

Загрузим все предобработанные данные 

In [4]:
X_train =pd.read_csv('../data/X_train.csv')
X_test =pd.read_csv('../data/X_test.csv')
y_train =pd.read_csv('../data/y_train.csv')
y_test =pd.read_csv('../data/y_test.csv')

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((4236, 570), (1060, 570), (4236, 1), (1060, 1))

Когда переводил в тензоры выдавало ошибку, потому что нужны флоат, а там видимо от ohe остались bool, переведем это все в флоат и пойдем дальше

In [5]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)
y_train = y_train.astype(float)
y_test = y_test.astype(float)

Переведем все в тензоры

In [6]:
X_train_tensor =torch.FloatTensor(X_train.values)
X_test_tensor =torch.FloatTensor(X_test.values)
y_train_tensor =torch.FloatTensor(y_train.values)
y_test_tensor =torch.FloatTensor(y_test.values)

X_train_tensor.shape, y_train_tensor.shape

(torch.Size([4236, 570]), torch.Size([4236, 1]))

Теперь соединим это обратно в датасеты (train/test), чтобы передавать в модели именно все признаки с таргетом, после чего создадим dataloader чтобы подавать в модель данные батчами/частями

In [7]:
train_ds = TensorDataset(X_train_tensor,y_train_tensor)
test_ds = TensorDataset(X_test_tensor,y_test_tensor)

train_ds

In [8]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True) #shuffle тут чтобы между эпохами перемешивались строки и модель не привыкала к порядку
test_loader = DataLoader(test_ds, batch_size=256)


In [9]:
input_size = X_train.shape[1]
input_size 

570

Сейчас построим достаточно базовую модель (вдохновление от 15 семинара). Архитектура простая - 3 полносвязных слоя и все. Далее будем строить несколько более сложных архитектур. Сначала простую чтобы понимать, помогают ли нам новые архитектуры улучшить так скажем 'базовое' качество. 

In [10]:
class Simple(nn.Module): # наследуемся от класса nn.Module
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 128) # первый скрытый слой - 128 нейронов
        self.fc2 = nn.Linear(128, 64) # второй скрытый - 64
        self.fc3 = nn.Linear(64, 1) # третий  выходной- 1 прогноз
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x) 
        return x

Теперь построим более сложную (чуть) архитектуру. Попробуем добавить просто больше скрытых слоев и добавим еще больше нейронов 

Но, при этом, есильно выше риск переобучения, поэтому будем все это сравнивать на качестве теста

In [11]:
class Deep(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 512) #теперь начинаем уменьшение размерности с 512 
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

И, добавим третью модель, тут мы добавим еще дропаут и батчнорм. Первый будет бороться с переобучением за счет отключения определенного количества нейронов во время обучения. Батчнорм будет просто стабилизировать значения внутри срктытых слоев

In [12]:
class Regularized(nn.Module):
    def __init__(self, input_size):
        super().__init__()


        self.net = nn.Sequential(nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.05), #1 скрытый 
                                 nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),nn.Dropout(0.05),  #2 скрытый
                                 nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))    #3 скрытый и 4 выходной
        #sequential тут потому что во первых мы так работали на семинарах, а во-вторых потому что слои и операции в них просто идут по порядку
        #без сложностей , и следующая функция красивая в 2 строки получается )))

    def forward(self, x):
        return self.net(x)

Мы решили добавить еще одну полнсвязную сеть с residual connection. Его преимущество в том, что он сохраняет вход блока и добавляет к выходу. То есь, это хорошо потому что мы по идее не теряем юзфул информацию когда проходим несколько слоев. В нашем случае, это полезно потому что после ohe у нас стало много признаков, и там есть важные признаки ,которые residual mlp поможет учитывать в каких то сложных комбинациях

Сайты, в том числе откуда и вдохновение на код:
- https://discuss.pytorch.org/t/how-to-use-residual-learning-applied-to-fully-connected-networks/98708/5
- https://stackoverflow.com/questions/60817390/implementing-a-simple-resnet-block-with-pytorch

In [13]:
class ResBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        
    def forward(self, x):
        residual = x  # сохраняем вход блока
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        x = x + residual  # прибавляем старый x к новому x
        x = torch.relu(x)
        
        return x

In [14]:
class ResMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 256)
        self.block1 = ResBlock(256)
        self.block2 = ResBlock(256)
        self.fc2 = nn.Linear(256, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.block1(x)
        x = self.block2(x)
        x = self.fc2(x)
        return x

Далее, переходим к следующей части , пока просто зададим функцию потерь и потом напишем фукнцию обучения

In [15]:
criterion = nn.MSELoss()

In [16]:
from tqdm import tqdm

def train(model, device, train_loader, optimizer, criterion, epoch): 
    model.train()
    train_loss = 0
    
    for data, target in tqdm(train_loader):
        data = data.to(device)
        target = target.to(device)
        
        optimizer.zero_grad() #обнуяем градиенты
        
        output = model(data) #прямой проход
        loss = criterion(output, target) #считаем ошибку
        
        loss.backward() #обратынй проход
        optimizer.step() #шаг оптимизатором
        train_loss += loss.item()
    
    train_loss = train_loss / len(train_loader)
    
    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(train_loss)) #как в семе, но чуть поправленная, потому что там для картинок
    return train_loss

Теперь функция тестирования, оч похожа на сем, но там картинки

In [17]:
def test(model, device, test_loader, criterion):
    model.eval()  
    
    test_loss = 0
    preds = []
    true = []
    
    # показываем, что обучения нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            test_loss += criterion(output, target).item()
            
            preds.append(output.cpu())
            true.append(target.cpu())
    
    test_loss = test_loss / len(test_loader)
    
    preds = torch.cat(preds).numpy().ravel() #тут мы клеим прогнозы в одномерный массив нампай 
    true = torch.cat(true).numpy().ravel()
    
    preds = np.expm1(preds) #возвращаем логарифмированные числа в первоначальное 
    true = np.expm1(true)
    
    mae = mean_absolute_error(true, preds)
    rmse = np.sqrt(mean_squared_error(true, preds))
    r2 = r2_score(true, preds)
    
    tqdm.write('Test set: Average loss: {:.4f}, MAE: {:.2f}, RMSE: {:.2f}, R2: {:.4f}'.format(
        test_loss, mae, rmse, r2))
    
    return test_loss, mae, rmse, r2

Теперь нужна функция запуска наших экспериментов, и после этого можно будет лицезреть качество выстроенных нами моделей))

In [18]:
# # основная функция для экспериментов
# def main(model, model_name):
#     seed_everything(42)  # фиксируем сиды
    
#     model = model.to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
#     train_losses = []
    
#     for epoch in range(1, 16):
#         print('\nEpoch:', epoch)
#         train_loss = train(model, device, train_loader, optimizer, criterion, epoch)
#         train_losses.append(train_loss)
#         test_loss, mae, rmse, r2 = test(model, device, test_loader, criterion)
    
#     print('Training is ended!')
    
#     result = {'model': model_name, 'test_loss': test_loss,'MAE': mae,'RMSE': rmse, 'R2': r2}
    
#     return model, train_losses, result

In [19]:
# основная функция для экспериментов
def main(model):
    seed_everything(42)  # фиксируем сиды
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    train_losses = []
    test_losses = []

    for epoch in range(1, 16):
        print('\nEpoch:', epoch)
        train_loss = train(model, device, train_loader, optimizer, criterion, epoch)
        test_loss, mae, rmse, r2 = test(model, device, test_loader, criterion)
        train_losses.append(train_loss)
        test_losses.append(test_loss)
    
    print('Training is ended!')
    
    
    # return model, train_losses, test_losses, test_loss, mae, rmse, r2

In [20]:
main(Simple(input_size))


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 874.51it/s]



Train set: Average loss: 154.6181
Test set: Average loss: 6.3434, MAE: 175640192.00, RMSE: 442042511.01, R2: -1896.2131

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 1193.21it/s]



Train set: Average loss: 3.1786
Test set: Average loss: 1.8145, MAE: 11987410.00, RMSE: 21867715.81, R2: -3.6430

Epoch: 3


...


Train set: Average loss: 0.0873
Test set: Average loss: 0.2604, MAE: 2957542.75, RMSE: 6070008.05, R2: 0.6423

Epoch: 14


100%|██████████| 67/67 [00:00<00:00, 1211.64it/s]



Train set: Average loss: 0.0755
Test set: Average loss: 0.2579, MAE: 2695520.50, RMSE: 5436531.44, R2: 0.7130

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 1210.64it/s]


Train set: Average loss: 0.0675
Test set: Average loss: 0.2489, MAE: 2573330.25, RMSE: 5338706.86, R2: 0.7233
Training is ended!


In [21]:
main(Deep(input_size))


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 580.45it/s]



Train set: Average loss: 71.0038
Test set: Average loss: 1.7153, MAE: 21532352.00, RMSE: 52541648.79, R2: -25.8037

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 652.20it/s]



Train set: Average loss: 0.9890
Test set: Average loss: 0.7606, MAE: 7256099.00, RMSE: 16550735.84, R2: -1.6596

Epoch: 3


...


Train set: Average loss: 0.0540
Test set: Average loss: 0.2162, MAE: 3020136.75, RMSE: 7526339.26, R2: 0.4500

Epoch: 14


100%|██████████| 67/67 [00:00<00:00, 638.06it/s]



Train set: Average loss: 0.0541
Test set: Average loss: 0.2134, MAE: 2470277.25, RMSE: 4654269.36, R2: 0.7897

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 691.31it/s]


Train set: Average loss: 0.0488
Test set: Average loss: 0.2200, MAE: 2552302.50, RMSE: 6217340.11, R2: 0.6247
Training is ended!


In [22]:
main(Regularized(input_size))


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 594.02it/s]



Train set: Average loss: 103.7855
Test set: Average loss: 2.5298, MAE: 8499811.00, RMSE: 12250580.68, R2: -0.4571

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 612.39it/s]



Train set: Average loss: 1.6608
Test set: Average loss: 0.7043, MAE: 5846094.00, RMSE: 9236522.69, R2: 0.1717

Epoch: 3


...


Train set: Average loss: 0.5271
Test set: Average loss: 0.2942, MAE: 4380681.50, RMSE: 7447857.45, R2: 0.4614

Epoch: 14


100%|██████████| 67/67 [00:00<00:00, 591.61it/s]



Train set: Average loss: 0.5420
Test set: Average loss: 0.2172, MAE: 4681803.50, RMSE: 9688129.32, R2: 0.0887

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 566.17it/s]


Train set: Average loss: 0.5243
Test set: Average loss: 0.2380, MAE: 3826799.00, RMSE: 6630608.67, R2: 0.5731
Training is ended!


In [23]:
main(ResMLP(input_size))


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 411.84it/s]



Train set: Average loss: 43.8960
Test set: Average loss: 0.8779, MAE: 12304962.00, RMSE: 41029536.93, R2: -15.3449

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 512.22it/s]



Train set: Average loss: 0.5527
Test set: Average loss: 0.4417, MAE: 5516935.50, RMSE: 13631449.23, R2: -0.8041

Epoch: 3


...


Train set: Average loss: 0.0396
Test set: Average loss: 0.1877, MAE: 3386257.00, RMSE: 6006796.22, R2: 0.6497

Epoch: 14


100%|██████████| 67/67 [00:00<00:00, 511.19it/s]



Train set: Average loss: 0.0406
Test set: Average loss: 0.1666, MAE: 2441461.25, RMSE: 4501166.46, R2: 0.8033

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 508.30it/s]


Train set: Average loss: 0.0366
Test set: Average loss: 0.1667, MAE: 2407299.50, RMSE: 5291020.19, R2: 0.7282
Training is ended!
